# Case Study: Complete Match Prediction Pipeline (EXTRA)

**Chapter 8: Feature Engineering for Soccer Predictions**

## Learning Objectives

- Apply all feature engineering techniques to a real prediction problem
- Build features for a specific match prediction
- Combine multiple data sources and feature types
- Create an interpretable prediction with feature explanations
- Understand the complete workflow from raw data to prediction
- Build a reusable prediction framework

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("Match Prediction Toolkit Ready!")

## The Case Study: Predicting USA vs. Netherlands

Imagine we're predicting the **2023 Women's World Cup Final** between USA and Netherlands.

We'll:
1. Generate historical tournament data
2. Engineer all features for both teams
3. Train a prediction model
4. Make the prediction with confidence intervals
5. Explain the prediction using feature importance

## Step 1: Generate Tournament Data

In [ ]:
# Generate Women's World Cup tournament data
np.random.seed(42)

teams = ['USA', 'Netherlands', 'England', 'Spain', 'Germany', 'France', 
         'Brazil', 'Argentina', 'Japan', 'Australia', 'Sweden', 'Norway',
         'Canada', 'Italy', 'Portugal', 'Mexico', 'South Korea', 'China',
         'New Zealand', 'Switzerland']

# Assign team strengths (for realistic simulation)
team_strength = {
    'USA': 0.9, 'Netherlands': 0.85, 'England': 0.82, 'Spain': 0.80,
    'Germany': 0.78, 'France': 0.76, 'Brazil': 0.74, 'Sweden': 0.72,
    'Japan': 0.70, 'Australia': 0.68, 'Norway': 0.66, 'Canada': 0.64,
    'Italy': 0.60, 'Argentina': 0.58, 'Portugal': 0.56, 'Mexico': 0.54,
    'South Korea': 0.52, 'China': 0.50, 'New Zealand': 0.48, 'Switzerland': 0.46
}

matches = []
match_id = 1
start_date = datetime(2023, 7, 20)

# Generate group stage and knockout matches
for i, home_team in enumerate(teams):
    for j, away_team in enumerate(teams):
        if i < j:  # Each pair plays once
            match_date = start_date + timedelta(days=match_id * 3)
            
            # Simulate scores based on team strength
            home_strength = team_strength[home_team]
            away_strength = team_strength[away_team]
            
            home_score = np.random.poisson(home_strength * 2)
            away_score = np.random.poisson(away_strength * 1.8)
            
            # Simulate detailed stats
            home_shots = int(home_strength * 15)
            away_shots = int(away_strength * 13)
            home_xg = home_shots * home_strength * 0.12
            away_xg = away_shots * away_strength * 0.10
            
            home_possession = 50 + (home_strength - away_strength) * 20
            away_possession = 100 - home_possession
            
            home_tackles = int((1 - home_strength) * 20)
            away_tackles = int((1 - away_strength) * 20)
            home_interceptions = int((1 - home_strength) * 12)
            away_interceptions = int((1 - away_strength) * 12)
            
            home_passes = int(home_possession * 10)
            away_passes = int(away_possession * 10)
            
            matches.append({
                'match_id': match_id,
                'match_date': match_date,
                'home_team': home_team,
                'away_team': away_team,
                'home_score': home_score,
                'away_score': away_score,
                'home_shots': home_shots,
                'away_shots': away_shots,
                'home_xg': home_xg,
                'away_xg': away_xg,
                'home_possession': home_possession,
                'away_possession': away_possession,
                'home_tackles': home_tackles,
                'away_tackles': away_tackles,
                'home_interceptions': home_interceptions,
                'away_interceptions': away_interceptions,
                'home_passes': home_passes,
                'away_passes': away_passes
            })
            
            match_id += 1

matches_df = pd.DataFrame(matches)
matches_df = matches_df.sort_values('match_date').reset_index(drop=True)

print(f"Generated {len(matches_df)} tournament matches")
print(f"Tournament period: {matches_df['match_date'].min()} to {matches_df['match_date'].max()}")
print(f"\nSample matches:")
matches_df.head(10)

## Step 2: Feature Engineering for All Matches

In [ ]:
def engineer_all_features(matches_df):
    """
    Complete feature engineering pipeline.
    Returns a dataframe with all features for each match.
    """
    
    print("Engineering features...\n")
    
    # 1. Calculate Elo ratings
    print("1. Calculating Elo ratings...")
    elo_ratings = {}
    elo_features = []
    
    for _, match in matches_df.sort_values('match_date').iterrows():
        home, away = match['home_team'], match['away_team']
        
        if home not in elo_ratings:
            elo_ratings[home] = 1500
        if away not in elo_ratings:
            elo_ratings[away] = 1500
        
        expected_home = 1 / (1 + 10 ** ((elo_ratings[away] - elo_ratings[home]) / 400))
        expected_away = 1 - expected_home
        
        elo_features.append({
            'match_id': match['match_id'],
            'home_elo': elo_ratings[home],
            'away_elo': elo_ratings[away],
            'elo_diff': elo_ratings[home] - elo_ratings[away]
        })
        
        if match['home_score'] > match['away_score']:
            actual_home, actual_away = 1, 0
        elif match['home_score'] < match['away_score']:
            actual_home, actual_away = 0, 1
        else:
            actual_home, actual_away = 0.5, 0.5
        
        elo_ratings[home] += 30 * (actual_home - expected_home)
        elo_ratings[away] += 30 * (actual_away - expected_away)
    
    elo_df = pd.DataFrame(elo_features)
    
    # 2. Calculate rolling averages
    print("2. Calculating rolling averages...")
    team_stats = []
    
    for _, match in matches_df.iterrows():
        # Home team stats
        team_stats.append({
            'match_id': match['match_id'],
            'team': match['home_team'],
            'venue': 'home',
            'goals': match['home_score'],
            'xg': match['home_xg'],
            'shots': match['home_shots'],
            'possession': match['home_possession'],
            'tackles': match['home_tackles']
        })
        # Away team stats
        team_stats.append({
            'match_id': match['match_id'],
            'team': match['away_team'],
            'venue': 'away',
            'goals': match['away_score'],
            'xg': match['away_xg'],
            'shots': match['away_shots'],
            'possession': match['away_possession'],
            'tackles': match['away_tackles']
        })
    
    stats_df = pd.DataFrame(team_stats)
    stats_df = stats_df.merge(matches_df[['match_id', 'match_date']], on='match_id')
    stats_df = stats_df.sort_values(['team', 'match_date'])
    
    # Rolling features
    for col in ['goals', 'xg', 'shots', 'possession', 'tackles']:
        stats_df[f'{col}_rolling_3'] = stats_df.groupby('team')[col].transform(
            lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)
        )
        stats_df[f'{col}_rolling_5'] = stats_df.groupby('team')[col].transform(
            lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
        )
    
    # 3. Calculate form (points from last matches)
    print("3. Calculating form...")
    stats_df['points'] = 0
    for idx, row in stats_df.iterrows():
        match = matches_df[matches_df['match_id'] == row['match_id']].iloc[0]
        if row['venue'] == 'home':
            if match['home_score'] > match['away_score']:
                stats_df.at[idx, 'points'] = 3
            elif match['home_score'] == match['away_score']:
                stats_df.at[idx, 'points'] = 1
        else:
            if match['away_score'] > match['home_score']:
                stats_df.at[idx, 'points'] = 3
            elif match['away_score'] == match['home_score']:
                stats_df.at[idx, 'points'] = 1
    
    stats_df['points_last_3'] = stats_df.groupby('team')['points'].transform(
        lambda x: x.rolling(window=3, min_periods=1).sum().shift(1)
    )
    stats_df['points_last_5'] = stats_df.groupby('team')['points'].transform(
        lambda x: x.rolling(window=5, min_periods=1).sum().shift(1)
    )
    
    print("✅ Feature engineering complete!\n")
    
    return elo_df, stats_df

elo_df, stats_df = engineer_all_features(matches_df)

## Step 3: Prepare Training Data

In [ ]:
# Merge features for each match
feature_data = []

for _, match in matches_df.iterrows():
    match_id = match['match_id']
    
    # Get home team features
    home_stats = stats_df[(stats_df['match_id'] == match_id) & 
                          (stats_df['team'] == match['home_team'])]
    
    # Get away team features
    away_stats = stats_df[(stats_df['match_id'] == match_id) & 
                          (stats_df['team'] == match['away_team'])]
    
    # Get Elo features
    elo_features = elo_df[elo_df['match_id'] == match_id]
    
    if len(home_stats) > 0 and len(away_stats) > 0 and len(elo_features) > 0:
        features = {
            'match_id': match_id,
            'home_team': match['home_team'],
            'away_team': match['away_team'],
            'home_score': match['home_score'],
            'away_score': match['away_score']
        }
        
        # Add home team rolling features
        for col in home_stats.columns:
            if 'rolling' in col or col == 'points_last_3' or col == 'points_last_5':
                features[f'home_{col}'] = home_stats[col].values[0]
        
        # Add away team rolling features
        for col in away_stats.columns:
            if 'rolling' in col or col == 'points_last_3' or col == 'points_last_5':
                features[f'away_{col}'] = away_stats[col].values[0]
        
        # Add Elo features
        features['home_elo'] = elo_features['home_elo'].values[0]
        features['away_elo'] = elo_features['away_elo'].values[0]
        features['elo_diff'] = elo_features['elo_diff'].values[0]
        
        # Target: home win
        features['home_win'] = 1 if match['home_score'] > match['away_score'] else 0
        
        feature_data.append(features)

feature_df = pd.DataFrame(feature_data).dropna()

print(f"Training dataset: {len(feature_df)} matches")
print(f"Features: {len([c for c in feature_df.columns if c not in ['match_id', 'home_team', 'away_team', 'home_score', 'away_score', 'home_win']])}")
print(f"\nTarget distribution:")
print(feature_df['home_win'].value_counts())
feature_df.head()

## Step 4: Train Prediction Model

In [ ]:
# Prepare features and target
feature_cols = [c for c in feature_df.columns if c not in 
                ['match_id', 'home_team', 'away_team', 'home_score', 'away_score', 'home_win']]

X = feature_df[feature_cols]
y = feature_df['home_win']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")

In [ ]:
# Train multiple models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    accuracy = (y_pred == y_test).mean()
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'auc': auc
    }
    
    print(f"  Accuracy: {accuracy:.3f}")
    print(f"  AUC: {auc:.3f}")

# Select best model
best_model_name = max(results, key=lambda x: results[x]['auc'])
best_model = results[best_model_name]['model']

print(f"\n✅ Best model: {best_model_name}")
print(f"   Accuracy: {results[best_model_name]['accuracy']:.3f}")
print(f"   AUC: {results[best_model_name]['auc']:.3f}")

## Step 5: Feature Importance Analysis

In [ ]:
# Get feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Top 15 Most Important Features:")
    print(importance_df.head(15))
    
    # Visualize
    plt.figure(figsize=(10, 8))
    top_features = importance_df.head(15)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Top 15 Features - {best_model_name}', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## Step 6: Predict USA vs. Netherlands Final

In [ ]:
# Get latest features for USA and Netherlands
usa_latest = stats_df[stats_df['team'] == 'USA'].iloc[-1]
ned_latest = stats_df[stats_df['team'] == 'Netherlands'].iloc[-1]

print("USA Recent Form:")
print(f"  Goals (last 5): {usa_latest['goals_rolling_5']:.2f}")
print(f"  xG (last 5): {usa_latest['xg_rolling_5']:.2f}")
print(f"  Points (last 5): {usa_latest['points_last_5']:.0f}/15")
print(f"  Current Elo: {elo_ratings['USA']:.0f}")

print("\nNetherlands Recent Form:")
print(f"  Goals (last 5): {ned_latest['goals_rolling_5']:.2f}")
print(f"  xG (last 5): {ned_latest['xg_rolling_5']:.2f}")
print(f"  Points (last 5): {ned_latest['points_last_5']:.0f}/15")
print(f"  Current Elo: {elo_ratings['Netherlands']:.0f}")

In [ ]:
# Create feature vector for the final
final_features = {}

# Home team (USA) features
for col in usa_latest.index:
    if 'rolling' in col or 'points_last' in col:
        final_features[f'home_{col}'] = usa_latest[col]

# Away team (Netherlands) features
for col in ned_latest.index:
    if 'rolling' in col or 'points_last' in col:
        final_features[f'away_{col}'] = ned_latest[col]

# Elo features
final_features['home_elo'] = elo_ratings['USA']
final_features['away_elo'] = elo_ratings['Netherlands']
final_features['elo_diff'] = elo_ratings['USA'] - elo_ratings['Netherlands']

# Create dataframe
final_df = pd.DataFrame([final_features])

# Ensure all features are present (fill missing with 0)
for col in feature_cols:
    if col not in final_df.columns:
        final_df[col] = 0

final_df = final_df[feature_cols]

print("\nFeature vector prepared for USA vs Netherlands")
print(f"Total features: {len(final_df.columns)}")

In [ ]:
# Make prediction
usa_win_probability = best_model.predict_proba(final_df)[0, 1]
ned_win_probability = 1 - usa_win_probability

print("="*60)
print("PREDICTION: USA vs. Netherlands (Final)")
print("="*60)
print(f"\nUSA Win Probability: {usa_win_probability:.1%}")
print(f"Netherlands Win Probability: {ned_win_probability:.1%}")
print(f"\nPredicted Winner: {'USA' if usa_win_probability > 0.5 else 'Netherlands'}")
print(f"Confidence: {max(usa_win_probability, ned_win_probability):.1%}")
print("="*60)

In [ ]:
# Visualize prediction
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Probability comparison
teams = ['USA', 'Netherlands']
probabilities = [usa_win_probability, ned_win_probability]
colors = ['#002868', '#FF6C00']  # USA and Netherlands colors

ax1.bar(teams, probabilities, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('Win Probability', fontsize=12)
ax1.set_title('Predicted Win Probabilities', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 1)
for i, (team, prob) in enumerate(zip(teams, probabilities)):
    ax1.text(i, prob + 0.05, f'{prob:.1%}', ha='center', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Key metrics comparison
metrics = ['Elo Rating', 'Goals (L5)', 'xG (L5)', 'Points (L5)']
usa_metrics = [
    elo_ratings['USA'],
    usa_latest['goals_rolling_5'],
    usa_latest['xg_rolling_5'],
    usa_latest['points_last_5']
]
ned_metrics = [
    elo_ratings['Netherlands'],
    ned_latest['goals_rolling_5'],
    ned_latest['xg_rolling_5'],
    ned_latest['points_last_5']
]

# Normalize for visualization
usa_norm = [v / max(usa_metrics[i], ned_metrics[i]) for i, v in enumerate(usa_metrics)]
ned_norm = [v / max(usa_metrics[i], ned_metrics[i]) for i, v in enumerate(ned_metrics)]

x = np.arange(len(metrics))
width = 0.35

ax2.barh(x - width/2, usa_norm, width, label='USA', color='#002868', alpha=0.7)
ax2.barh(x + width/2, ned_norm, width, label='Netherlands', color='#FF6C00', alpha=0.7)
ax2.set_yticks(x)
ax2.set_yticklabels(metrics)
ax2.set_xlabel('Normalized Value', fontsize=12)
ax2.set_title('Key Metrics Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Summary

In this case study, you learned:

✅ **Complete prediction pipeline** - from raw data to final prediction

✅ **Feature engineering at scale** - combining multiple feature types

✅ **Model training and selection** - comparing multiple algorithms

✅ **Feature importance** - understanding what drives predictions

✅ **Interpretable predictions** - explaining the "why" behind predictions

### Key Insights

**1. Features matter more than models**
- Well-engineered features improve all models
- Simple models with great features often beat complex models with poor features

**2. Multiple perspectives**
- Elo captures current strength
- Rolling averages capture recent form
- Points capture momentum
- All contribute to better predictions

**3. Interpretability is crucial**
- Feature importance helps explain predictions
- Stakeholders want to know "why"
- Builds trust in the model

**4. Continuous improvement**
- Monitor model performance
- Update features as new data arrives
- Retrain models regularly

## Practice Exercises

1. **Add more features**: Include set pieces, PPDA, Colley ratings

2. **Predict score**: Build a regression model to predict exact scores

3. **Confidence intervals**: Calculate prediction confidence intervals

4. **Feature interactions**: Create interaction features (e.g., Elo × recent form)

5. **Temporal validation**: Use time-based train/test split

6. **Ensemble methods**: Combine predictions from multiple models

7. **SHAP values**: Use SHAP for more detailed feature explanations

8. **Real-time updates**: Build a system that updates predictions as matches are played